# Notebook 09: The Quantum Fourier Transform

---

## Overview

The Quantum Fourier Transform (QFT) is the quantum analogue of the Discrete Fourier Transform (DFT). It is a **key subroutine** in many quantum algorithms, including Shor's factoring algorithm, quantum phase estimation, and quantum counting.

While the classical DFT on $N = 2^n$ points requires $O(N \log N)$ operations (via FFT), the QFT requires only $O(n^2) = O((\log N)^2)$ quantum gates — an **exponential speedup** in the circuit size.

### What You Will Learn

1. Classical DFT review and comparison with QFT
2. The QFT matrix and its mathematical structure
3. Step-by-step circuit decomposition
4. Phase estimation algorithm (QPE)
5. Applications: period finding, quantum counting
6. Full Qiskit implementations with verification

## 1. Classical Discrete Fourier Transform

### Definition

The DFT of a vector $(x_0, x_1, \ldots, x_{N-1})$ produces $(y_0, y_1, \ldots, y_{N-1})$ where:

$$y_k = \frac{1}{\sqrt{N}} \sum_{j=0}^{N-1} x_j \, \omega^{jk}, \quad \omega = e^{2\pi i / N}$$

In matrix form:

$$F_N = \frac{1}{\sqrt{N}} \begin{pmatrix} 1 & 1 & 1 & \cdots & 1 \\ 1 & \omega & \omega^2 & \cdots & \omega^{N-1} \\ 1 & \omega^2 & \omega^4 & \cdots & \omega^{2(N-1)} \\ \vdots & \vdots & \vdots & \ddots & \vdots \\ 1 & \omega^{N-1} & \omega^{2(N-1)} & \cdots & \omega^{(N-1)^2} \end{pmatrix}$$

### Properties

- $F_N$ is **unitary**: $F_N F_N^\dagger = I$
- **Complexity:** Direct computation is $O(N^2)$. The Fast Fourier Transform (FFT) reduces this to $O(N \log N)$.

### Applications

The DFT is ubiquitous: signal processing, image compression (JPEG), data analysis, convolution, polynomial multiplication, and more.

## 2. The Quantum Fourier Transform

### Definition

The QFT acts on computational basis states as:

$$\text{QFT}|j\rangle = \frac{1}{\sqrt{N}} \sum_{k=0}^{N-1} e^{2\pi i jk/N} |k\rangle$$

where $N = 2^n$ and $j, k \in \{0, 1, \ldots, N-1\}$.

On a general state $|\psi\rangle = \sum_j x_j |j\rangle$:

$$\text{QFT}|\psi\rangle = \sum_k y_k |k\rangle, \quad y_k = \frac{1}{\sqrt{N}} \sum_{j=0}^{N-1} x_j e^{2\pi i jk/N}$$

This is exactly the DFT of the amplitudes!

### Key Difference from Classical DFT

- The classical DFT transforms a vector of $N$ numbers $\to$ vector of $N$ numbers.
- The QFT transforms amplitudes of $n$ qubits $\to$ amplitudes of $n$ qubits.
- We cannot directly read out all the transformed amplitudes (measurement gives one sample).
- The power comes from using QFT as a **subroutine** inside larger algorithms.

## 3. QFT in Product Form — The Circuit Decomposition

The key to building the QFT circuit is the **product representation**.

Write $j$ in binary: $j = j_1 j_2 \cdots j_n$ (where $j = j_1 2^{n-1} + j_2 2^{n-2} + \cdots + j_n 2^0$).

Similarly, write $k$ in binary. Then:

$$e^{2\pi i jk/2^n} = e^{2\pi i j \cdot 0.k_1 k_2 \cdots k_n}$$

where $0.k_1 k_2 \cdots k_n = k_1/2 + k_2/4 + \cdots + k_n/2^n$ is the binary fraction.

The QFT can be written as:

$$\text{QFT}|j_1 j_2 \cdots j_n\rangle = \frac{1}{\sqrt{2^n}} \bigotimes_{l=1}^{n} \left( |0\rangle + e^{2\pi i j \cdot 0.j_l j_{l+1} \cdots j_n} |1\rangle \right)$$

This is a **tensor product** of single-qubit states! Each qubit $l$ acquires a phase that depends on bits $j_l, j_{l+1}, \ldots, j_n$.

### For 3 qubits:

$$\text{QFT}|j_1 j_2 j_3\rangle = \frac{1}{2\sqrt{2}} \left(|0\rangle + e^{2\pi i \cdot 0.j_3}|1\rangle\right) \otimes \left(|0\rangle + e^{2\pi i \cdot 0.j_2 j_3}|1\rangle\right) \otimes \left(|0\rangle + e^{2\pi i \cdot 0.j_1 j_2 j_3}|1\rangle\right)$$

Note the **bit reversal**: the output qubit ordering is reversed compared to the input.

## 4. The QFT Circuit

### Gates Used

1. **Hadamard gate** $H$: Creates superposition
$$H|j_l\rangle = \frac{1}{\sqrt{2}}\left(|0\rangle + e^{2\pi i \cdot 0.j_l}|1\rangle\right)$$

(Since $e^{2\pi i \cdot j_l/2} = e^{\pi i j_l} = (-1)^{j_l}$, this is just $H$.)

2. **Controlled phase gates** $R_k$:
$$R_k = \begin{pmatrix} 1 & 0 \\ 0 & e^{2\pi i / 2^k} \end{pmatrix}$$

The controlled-$R_k$ gate adds the phase $e^{2\pi i / 2^k}$ to $|1\rangle$ of the target qubit, controlled on another qubit being $|1\rangle$.

### Circuit for $n$ qubits:

For qubit $l$ (from 1 to $n$):
1. Apply $H$ to qubit $l$
2. Apply controlled-$R_2$ (control: qubit $l+1$, target: qubit $l$)
3. Apply controlled-$R_3$ (control: qubit $l+2$, target: qubit $l$)
4. $\ldots$ continue until controlled-$R_{n-l+1}$

After all qubits are processed, apply SWAP gates to reverse the qubit order.

### Gate Count

- $n$ Hadamard gates
- $n(n-1)/2$ controlled phase gates
- $\lfloor n/2 \rfloor$ SWAP gates
- **Total:** $O(n^2)$ gates = $O((\log N)^2)$

Compare: classical FFT needs $O(N \log N) = O(2^n \cdot n)$ operations.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, Operator
from qiskit.visualization import plot_histogram, plot_bloch_multivector
import matplotlib.pyplot as plt

print("Imports successful!")

In [ ]:
def qft_circuit(n, with_swaps=True, inverse=False):
    """
    Build the Quantum Fourier Transform circuit for n qubits.
    
    Parameters:
    -----------
    n : int — number of qubits
    with_swaps : bool — include final SWAP gates for bit reversal
    inverse : bool — if True, build QFT† (inverse QFT)
    
    Returns: QuantumCircuit
    """
    qc = QuantumCircuit(n, name='QFT' if not inverse else 'QFT†')
    
    if not inverse:
        # Forward QFT
        for j in range(n):
            qc.h(j)
            for k in range(j + 1, n):
                angle = np.pi / (2**(k - j))
                qc.cp(angle, k, j)
        
        # Swap qubits for bit reversal
        if with_swaps:
            for i in range(n // 2):
                qc.swap(i, n - 1 - i)
    else:
        # Inverse QFT = reverse the circuit
        if with_swaps:
            for i in range(n // 2):
                qc.swap(i, n - 1 - i)
        
        for j in range(n - 1, -1, -1):
            for k in range(n - 1, j, -1):
                angle = -np.pi / (2**(k - j))
                qc.cp(angle, k, j)
            qc.h(j)
    
    return qc

# Build and display QFT for 3 qubits
qft3 = qft_circuit(3)
qft3.draw('mpl', style='iqp', fold=False)
plt.title('Quantum Fourier Transform (3 qubits)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Display QFT for 4 qubits
qft4 = qft_circuit(4)
qft4.draw('mpl', style='iqp', fold=False)
plt.title('Quantum Fourier Transform (4 qubits)', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Verifying the QFT Matrix

Let us verify that our circuit produces the correct QFT matrix by comparing with the classical DFT matrix.

In [ ]:
def classical_dft_matrix(n):
    """Construct the N×N DFT matrix where N = 2^n."""
    N = 2**n
    omega = np.exp(2j * np.pi / N)
    F = np.zeros((N, N), dtype=complex)
    for j in range(N):
        for k in range(N):
            F[j, k] = omega**(j * k) / np.sqrt(N)
    return F

# Compare QFT circuit matrix with classical DFT matrix
for n in [1, 2, 3, 4]:
    # Get QFT matrix from Qiskit circuit
    qft = qft_circuit(n, with_swaps=True)
    qft_matrix = Operator(qft).data
    
    # Classical DFT matrix
    dft_matrix = classical_dft_matrix(n)
    
    # Compare
    match = np.allclose(qft_matrix, dft_matrix)
    print(f"n={n} (N={2**n}): QFT circuit matches DFT matrix: {match}")

print("\nAll verified!")

In [ ]:
# Visualize the QFT matrix (magnitude and phase)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for idx, n in enumerate([2, 3, 4]):
    qft = qft_circuit(n)
    matrix = Operator(qft).data
    N = 2**n
    
    # Magnitude (should be uniform 1/√N)
    im1 = axes[0, idx].imshow(np.abs(matrix), cmap='Blues', vmin=0, vmax=0.6)
    axes[0, idx].set_title(f'|QFT| (n={n})', fontsize=12)
    axes[0, idx].set_xlabel('Column')
    axes[0, idx].set_ylabel('Row')
    plt.colorbar(im1, ax=axes[0, idx], shrink=0.8)
    
    # Phase
    phases = np.angle(matrix) / np.pi  # in units of π
    im2 = axes[1, idx].imshow(phases, cmap='hsv', vmin=-1, vmax=1)
    axes[1, idx].set_title(f'Phase/π of QFT (n={n})', fontsize=12)
    axes[1, idx].set_xlabel('Column')
    axes[1, idx].set_ylabel('Row')
    plt.colorbar(im2, ax=axes[1, idx], shrink=0.8)

plt.suptitle('QFT Matrix Structure: Magnitude and Phase', fontsize=14)
plt.tight_layout()
plt.show()

## 6. QFT on Specific Input States

Let us see what the QFT does to various input states.

In [ ]:
def apply_qft_and_visualize(n, input_state_fn, title):
    """
    Apply QFT to a given input state and visualize before/after.
    
    input_state_fn: function that takes a QuantumCircuit and prepares the input state
    """
    N = 2**n
    
    # Prepare input state
    qc_before = QuantumCircuit(n)
    input_state_fn(qc_before)
    sv_before = Statevector.from_instruction(qc_before)
    
    # Apply QFT
    qc_after = QuantumCircuit(n)
    input_state_fn(qc_after)
    qft = qft_circuit(n)
    qc_after.append(qft, range(n))
    sv_after = Statevector.from_instruction(qc_after)
    
    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    labels = [format(i, f'0{n}b') for i in range(N)]
    
    # Before QFT
    probs_before = np.abs(sv_before.data)**2
    ax1.bar(labels, probs_before, color='steelblue', alpha=0.8)
    ax1.set_title('Before QFT', fontsize=12)
    ax1.set_ylabel('Probability')
    ax1.set_ylim(0, 1.1)
    if n > 3:
        ax1.tick_params(axis='x', rotation=90, labelsize=7)
    
    # After QFT
    probs_after = np.abs(sv_after.data)**2
    ax2.bar(labels, probs_after, color='coral', alpha=0.8)
    ax2.set_title('After QFT', fontsize=12)
    ax2.set_ylabel('Probability')
    ax2.set_ylim(0, 1.1)
    if n > 3:
        ax2.tick_params(axis='x', rotation=90, labelsize=7)
    
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

# QFT of |0...0⟩ → uniform superposition
apply_qft_and_visualize(3, lambda qc: None, 'QFT of |000⟩ → uniform superposition')

# QFT of |1⟩ (state |001⟩ in 3 qubits)
apply_qft_and_visualize(3, lambda qc: qc.x(0), 'QFT of |001⟩')

# QFT of uniform superposition → |0...0⟩
def prepare_uniform(qc):
    for i in range(qc.num_qubits):
        qc.h(i)

apply_qft_and_visualize(3, prepare_uniform, 'QFT of uniform superposition → |000⟩')

In [ ]:
# QFT of various basis states
n = 3
N = 2**n

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for j in range(N):
    row = j // 4
    col = j % 4
    
    # Prepare |j⟩ and apply QFT
    qc = QuantumCircuit(n)
    # Set bits of j
    for bit in range(n):
        if (j >> bit) & 1:
            qc.x(bit)
    qft = qft_circuit(n)
    qc.append(qft, range(n))
    
    sv = Statevector.from_instruction(qc)
    
    # Plot amplitudes (real part shown with phase as color)
    probs = np.abs(sv.data)**2
    phases = np.angle(sv.data)
    
    labels = [format(i, f'0{n}b') for i in range(N)]
    colors = plt.cm.hsv((phases + np.pi) / (2 * np.pi))
    
    axes[row, col].bar(labels, probs, color=colors, alpha=0.9)
    axes[row, col].set_title(f'QFT|{format(j, f"0{n}b")}⟩', fontsize=11)
    axes[row, col].set_ylim(0, 0.3)
    axes[row, col].tick_params(axis='x', rotation=45, labelsize=7)

plt.suptitle('QFT of all 3-qubit basis states\n(bar height = probability, color = phase)', fontsize=14)
plt.tight_layout()
plt.show()

print("Note: All basis states map to uniform probability distributions!")
print("The difference is in the PHASES, which carry the frequency information.")

## 7. QFT for Period Finding

The most important application of QFT is **period finding**. If a state has periodic amplitudes with period $r$, the QFT reveals peaks at multiples of $N/r$.

$$|\psi\rangle = \frac{1}{\sqrt{A}} \sum_{k=0}^{A-1} |x_0 + kr\rangle \xrightarrow{\text{QFT}} \text{peaks at } m = j \cdot \frac{N}{r}$$

In [ ]:
def demonstrate_qft_period_finding(n, period, offset=0):
    """
    Create a periodic state and show that QFT reveals the period.
    """
    N = 2**n
    
    # Create periodic state manually
    state = np.zeros(N, dtype=complex)
    positions = list(range(offset, N, period))
    for pos in positions:
        if pos < N:
            state[pos] = 1.0
    state = state / np.linalg.norm(state)  # normalize
    
    # Apply QFT (as classical DFT on the amplitudes)
    dft_matrix = classical_dft_matrix(n)
    qft_state = dft_matrix @ state
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    labels = list(range(N))
    
    # Before QFT
    ax1.bar(labels, np.abs(state)**2, color='steelblue', alpha=0.8)
    ax1.set_title(f'Periodic state (period={period}, offset={offset})', fontsize=12)
    ax1.set_xlabel('State index')
    ax1.set_ylabel('Probability')
    
    # After QFT
    ax2.bar(labels, np.abs(qft_state)**2, color='coral', alpha=0.8)
    ax2.set_title(f'After QFT: peaks at multiples of N/r = {N}/{period} = {N//period}', fontsize=12)
    ax2.set_xlabel('Frequency index')
    ax2.set_ylabel('Probability')
    
    # Mark expected peaks
    for k in range(period):
        peak = k * N // period
        if peak < N:
            ax2.axvline(x=peak, color='red', linestyle='--', alpha=0.5)
    
    plt.suptitle(f'QFT Period Finding (n={n}, N={N})', fontsize=14)
    plt.tight_layout()
    plt.show()

# Period 4 in 16 states
demonstrate_qft_period_finding(4, period=4, offset=0)

# Period 4 with offset
demonstrate_qft_period_finding(4, period=4, offset=1)

# Period 2
demonstrate_qft_period_finding(4, period=2, offset=0)

# Period 8
demonstrate_qft_period_finding(4, period=8, offset=0)

## 8. Quantum Phase Estimation (QPE)

QPE is one of the most important quantum subroutines. Given a unitary $U$ and an eigenstate $|u\rangle$ with $U|u\rangle = e^{2\pi i \phi}|u\rangle$, QPE estimates the phase $\phi$.

### The Circuit

```
|0⟩ ── H ─── ●────────────────────── ┤       ├── Measure
|0⟩ ── H ────┼──── ●──────────────── ┤ QFT†  ├── Measure  
|0⟩ ── H ────┼─────┼──── ●────────── ┤       ├── Measure
             │     │     │
|u⟩ ──── U^(2²) U^(2¹) U^(2⁰) ──────────────────────────
```

### Mathematical Derivation

**Step 1:** After Hadamards on $t$ counting qubits:
$$\frac{1}{\sqrt{2^t}} \sum_{k=0}^{2^t - 1} |k\rangle |u\rangle$$

**Step 2:** After controlled-$U^{2^j}$ operations:
$$\frac{1}{\sqrt{2^t}} \sum_{k=0}^{2^t - 1} e^{2\pi i \phi k} |k\rangle |u\rangle$$

**Step 3:** The counting register is in state $\frac{1}{\sqrt{2^t}} \sum_{k=0}^{2^t - 1} e^{2\pi i \phi k} |k\rangle$ — this is exactly the QFT of $|2^t \phi\rangle$ (rounded to nearest integer).

**Step 4:** Apply QFT$^\dagger$ to get $|\tilde{\phi}\rangle$ where $\tilde{\phi} \approx 2^t \phi$.

**Step 5:** Measure to get $\tilde{\phi}$. Then $\phi \approx \tilde{\phi} / 2^t$.

In [ ]:
def quantum_phase_estimation(unitary_gate, eigenstate_prep, n_counting, n_work):
    """
    Build a Quantum Phase Estimation circuit.
    
    Parameters:
    -----------
    unitary_gate : Gate — the unitary U whose phase we want to estimate
    eigenstate_prep : function — prepares |u⟩ on the work register
    n_counting : int — number of counting qubits (precision)
    n_work : int — number of work qubits
    
    Returns: QuantumCircuit
    """
    total_qubits = n_counting + n_work
    qc = QuantumCircuit(total_qubits, n_counting)
    
    # Prepare eigenstate on work register
    eigenstate_prep(qc, list(range(n_counting, total_qubits)))
    
    # Hadamard on counting register
    for i in range(n_counting):
        qc.h(i)
    
    qc.barrier()
    
    # Controlled U^(2^j)
    for j in range(n_counting):
        # Apply U^(2^j) controlled on counting qubit j
        repetitions = 2**j
        cu = unitary_gate.control()
        for _ in range(repetitions):
            qc.append(cu, [j] + list(range(n_counting, total_qubits)))
    
    qc.barrier()
    
    # Inverse QFT on counting register
    qft_inv = qft_circuit(n_counting, inverse=True)
    qc.append(qft_inv, range(n_counting))
    
    qc.barrier()
    
    # Measure counting register
    for i in range(n_counting):
        qc.measure(i, i)
    
    return qc

print("QPE circuit builder defined.")

In [ ]:
# Example: Phase estimation for a T gate
# T gate: T|1⟩ = e^(iπ/4)|1⟩, so phase = 1/8

# Create the T gate as a unitary
t_gate = QuantumCircuit(1, name='T')
t_gate.t(0)
t_gate = t_gate.to_gate()

# Eigenstate: |1⟩
def prep_one(qc, qubits):
    qc.x(qubits[0])

# Build QPE circuit
n_counting = 4
qpe_circuit = quantum_phase_estimation(t_gate, prep_one, n_counting, 1)

print(f"QPE for T gate (expected phase = 1/8 = 0.125)")
print(f"Counting qubits: {n_counting}")

# Draw
qpe_circuit.draw('mpl', style='iqp', fold=30)
plt.title('QPE Circuit for T Gate', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Run QPE for T gate
sim = AerSimulator()
compiled = transpile(qpe_circuit, sim)
result = sim.run(compiled, shots=2048).result()
counts = result.get_counts()

print("QPE Results for T gate:")
print(f"Expected phase: 1/8 = 0.125")
print(f"Expected measurement: {int(0.125 * 2**n_counting)} (= 2 in decimal)")
print(f"Expected binary: {format(int(0.125 * 2**n_counting), f'0{n_counting}b')}")
print(f"\nMeasurement results: {counts}")

# Decode
for bitstring, count in sorted(counts.items(), key=lambda x: x[1], reverse=True):
    decimal = int(bitstring, 2)
    phase = decimal / 2**n_counting
    print(f"  |{bitstring}⟩ (decimal {decimal}): phase = {decimal}/{2**n_counting} = {phase:.6f}, count = {count}")

plot_histogram(counts)
plt.title('QPE for T gate: Phase = 1/8')
plt.show()

In [ ]:
# QPE for S gate: S|1⟩ = e^(iπ/2)|1⟩ = i|1⟩, phase = 1/4
s_gate = QuantumCircuit(1, name='S')
s_gate.s(0)
s_gate = s_gate.to_gate()

n_counting = 4
qpe_s = quantum_phase_estimation(s_gate, prep_one, n_counting, 1)

compiled = transpile(qpe_s, sim)
result = sim.run(compiled, shots=2048).result()
counts = result.get_counts()

print("QPE for S gate: expected phase = 1/4 = 0.25")
for bitstring, count in sorted(counts.items(), key=lambda x: x[1], reverse=True)[:3]:
    decimal = int(bitstring, 2)
    phase = decimal / 2**n_counting
    print(f"  |{bitstring}⟩ → phase = {phase:.4f}, count = {count}")

# QPE for Z gate: Z|1⟩ = -|1⟩ = e^(iπ)|1⟩, phase = 1/2
z_gate = QuantumCircuit(1, name='Z')
z_gate.z(0)
z_gate = z_gate.to_gate()

qpe_z = quantum_phase_estimation(z_gate, prep_one, n_counting, 1)
compiled = transpile(qpe_z, sim)
result = sim.run(compiled, shots=2048).result()
counts = result.get_counts()

print("\nQPE for Z gate: expected phase = 1/2 = 0.50")
for bitstring, count in sorted(counts.items(), key=lambda x: x[1], reverse=True)[:3]:
    decimal = int(bitstring, 2)
    phase = decimal / 2**n_counting
    print(f"  |{bitstring}⟩ → phase = {phase:.4f}, count = {count}")

## 9. QPE with Non-Exact Phases

When the phase $\phi$ cannot be represented exactly in $t$ bits (i.e., $2^t \phi$ is not an integer), the measurement distribution spreads around the closest representable value. More counting qubits $\to$ better precision.

In [ ]:
# QPE for a rotation gate with irrational phase
# R_z(θ)|1⟩ = e^(-iθ/2)|1⟩, so phase = -θ/(2·2π) mod 1 = 1 - θ/(4π)

theta = 2 * np.pi / 3  # phase = 1 - 1/6 = 5/6 ≈ 0.8333...
true_phase = 1 - theta / (2 * np.pi * 2)
# Actually, for Rz(θ)|1⟩ = e^{-iθ/2}|1⟩, phase = (-θ/2)/(2π) mod 1
true_phase = (-theta / 2) / (2 * np.pi) % 1

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, n_count in enumerate([3, 5, 8]):
    rz_gate = QuantumCircuit(1, name=f'Rz({theta:.2f})')
    rz_gate.rz(theta, 0)
    rz_gate = rz_gate.to_gate()
    
    qpe = quantum_phase_estimation(rz_gate, prep_one, n_count, 1)
    compiled = transpile(qpe, sim)
    result = sim.run(compiled, shots=4096).result()
    counts = result.get_counts()
    
    # Convert to phases
    phases = {}
    for bitstring, count in counts.items():
        decimal = int(bitstring, 2)
        phase = decimal / 2**n_count
        phases[phase] = phases.get(phase, 0) + count
    
    # Sort and plot
    sorted_phases = sorted(phases.items())
    x_vals = [p for p, _ in sorted_phases]
    y_vals = [c / 4096 for _, c in sorted_phases]
    
    axes[idx].bar(x_vals, y_vals, width=1/(2**n_count), color='steelblue', alpha=0.8)
    axes[idx].axvline(x=true_phase, color='red', linestyle='--', linewidth=2, label=f'True: {true_phase:.4f}')
    axes[idx].set_xlabel('Estimated phase', fontsize=11)
    axes[idx].set_ylabel('Probability', fontsize=11)
    axes[idx].set_title(f't={n_count} counting qubits', fontsize=12)
    axes[idx].legend(fontsize=10)
    axes[idx].set_xlim(-0.05, 1.05)

plt.suptitle(f'QPE Precision: Phase = {true_phase:.6f} (Rz gate)', fontsize=14)
plt.tight_layout()
plt.show()

## 10. Inverse QFT Verification

Let us verify that QFT followed by QFT$^\dagger$ gives the identity.

In [ ]:
# Verify QFT · QFT† = I
for n in [2, 3, 4, 5]:
    qc = QuantumCircuit(n)
    
    # Apply QFT then QFT†
    qft_fwd = qft_circuit(n, inverse=False)
    qft_inv = qft_circuit(n, inverse=True)
    qc.append(qft_fwd, range(n))
    qc.append(qft_inv, range(n))
    
    # Check if it is identity
    op = Operator(qc)
    is_identity = np.allclose(op.data, np.eye(2**n))
    print(f"n={n}: QFT · QFT† = I? {is_identity}")

print("\nAll verified — QFT† is the correct inverse!")

## 11. Approximate QFT

For large $n$, the controlled phase gates $R_k$ with very high $k$ (small angles) contribute negligibly. We can **truncate** these gates to get an approximate QFT with fewer gates.

An approximate QFT that keeps only controlled rotations up to $R_m$ (ignoring $R_{m+1}, R_{m+2}, \ldots$) has:
- Gate count: $O(nm)$ instead of $O(n^2)$
- Error: $O(n/2^m)$ (decreases exponentially with $m$)

For most applications, $m = O(\log n)$ suffices, giving an $O(n \log n)$ QFT!

In [ ]:
def approximate_qft(n, max_k=None):
    """
    Build an approximate QFT that truncates controlled rotations.
    
    Parameters:
    -----------
    n : int — number of qubits
    max_k : int — maximum k for R_k gates (None = exact QFT)
    """
    qc = QuantumCircuit(n, name=f'AQFT(k≤{max_k})')
    
    for j in range(n):
        qc.h(j)
        for k in range(j + 1, n):
            rotation_order = k - j
            if max_k is not None and rotation_order > max_k:
                continue  # Skip high-order rotations
            angle = np.pi / (2**rotation_order)
            qc.cp(angle, k, j)
    
    # Swap for bit reversal
    for i in range(n // 2):
        qc.swap(i, n - 1 - i)
    
    return qc

# Compare exact vs approximate QFT
n = 6
exact_qft = qft_circuit(n)
exact_matrix = Operator(exact_qft).data

print(f"Approximate QFT Error Analysis (n={n})")
print("=" * 50)
print(f"{'max_k':<8} {'# Gates':<12} {'Frobenius Error':<18} {'Max Entry Error'}")
print("-" * 50)

for max_k in [1, 2, 3, 4, 5, None]:
    aqft = approximate_qft(n, max_k)
    aqft_matrix = Operator(aqft).data
    
    error = np.linalg.norm(exact_matrix - aqft_matrix, 'fro')
    max_error = np.max(np.abs(exact_matrix - aqft_matrix))
    
    # Count gates
    compiled = transpile(aqft, basis_gates=['h', 'cx', 'rz'], optimization_level=0)
    n_gates = compiled.size()
    
    k_str = str(max_k) if max_k else 'Exact'
    print(f"{k_str:<8} {n_gates:<12} {error:<18.6f} {max_error:.6f}")

## 12. Applications Summary

The QFT appears in many quantum algorithms:

| Application | Uses QFT for... |
|-------------|----------------|
| **Shor's Algorithm** | Period finding via phase estimation |
| **Quantum Phase Estimation** | Extracting eigenvalue phases |
| **Quantum Counting** | Estimating number of solutions (with Grover) |
| **Hidden Subgroup Problem** | Identifying hidden structure in groups |
| **Quantum Simulation** | Efficient Hamiltonian decomposition |
| **HHL Algorithm** | Linear systems solver (eigenvalue inversion) |

In [ ]:
# Complexity comparison: QFT vs classical FFT
ns = list(range(2, 25))

# Classical FFT: O(N log N) = O(2^n * n)
fft_ops = [2**n * n for n in ns]

# Quantum QFT: O(n^2) gates
qft_ops = [n**2 for n in ns]

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(ns, fft_ops, 'ro-', linewidth=2, markersize=6, label='Classical FFT: $O(2^n \cdot n)$')
ax.semilogy(ns, qft_ops, 'bs-', linewidth=2, markersize=6, label='Quantum QFT: $O(n^2)$')

ax.set_xlabel('Number of qubits $n$ ($N = 2^n$ data points)', fontsize=13)
ax.set_ylabel('Number of operations', fontsize=13)
ax.set_title('QFT vs FFT: Gate/Operation Count', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Note: The QFT speedup is exponential in the number of operations.")
print("However, we cannot directly read all N Fourier coefficients —")
print("measurement gives only one sample. The power comes from using QFT")
print("as a subroutine (e.g., in phase estimation).")

## 13. Exercises

1. **Manual QFT:** Compute $\text{QFT}|101\rangle$ by hand using the product formula and verify with Qiskit.

2. **QFT vs Hadamard:** Show that for $n=1$, the QFT is just the Hadamard gate. For $n=2$, write out the QFT matrix and compare with $H^{\otimes 2}$.

3. **QPE precision:** If you need to estimate a phase to $b$ bits of accuracy with success probability $\geq 1 - \epsilon$, how many counting qubits $t$ do you need? (Answer: $t = b + \lceil \log_2(2 + 1/(2\epsilon)) \rceil$)

4. **Implement** QPE for a 2-qubit unitary of your choice. Verify that it correctly extracts both eigenvalues.

5. **Approximate QFT:** For $n = 10$, compare the fidelity (overlap with exact QFT output) of the approximate QFT with $m = 2, 3, 4, 5$. At what $m$ is the fidelity $> 0.99$?

## 14. Summary

| Concept | Details |
|---------|--------|
| **QFT** | Quantum analogue of DFT: $\text{QFT}|j\rangle = \frac{1}{\sqrt{N}} \sum_k e^{2\pi ijk/N} |k\rangle$ |
| **Gate count** | $O(n^2)$ for $n$ qubits (exponentially faster than FFT) |
| **Circuit** | Hadamards + controlled phase rotations + bit reversal SWAPs |
| **Phase Estimation** | Extracts eigenvalue phase of a unitary |
| **Applications** | Shor's algorithm, QPE, quantum counting, HHL |
| **Approximate QFT** | Truncate small rotations: $O(n \log n)$ gates with $O(n/2^m)$ error |

**Next:** [Notebook 10 — Variational Algorithms: VQE and QAOA](10-variational-algorithms-vqe-qaoa.ipynb)